## Split Overmerged Authors — Middle-Name Contradictions

Identifies work_authors where parsed **first** and **last** names match the author profile, but the **middle names are incompatible** — both middles are non-empty and either (a) both single-letter initials that disagree, or (b) both multi-char tokens whose first letters disagree, without substring overlap or token reorder.

Same plumbing as Phase 2 (per-row ORCID exclusion, minority-cluster guard, curated queue enqueue). New elements:

**Guards added for this phase:**
- **CJK + Arabic script exclusion** (extended from Phase 2's CJK-only range)
- **Profile parser-artifact regex** — drops `[UNESP]`-style affiliation tags, embedded dates, `AUTHOR`/`USGS`/`NULL` markers
- **Token-reorder guard** — `SORT_ARRAY(SPLIT) <> SORT_ARRAY(SPLIT)` drops same-middle-different-order cases
- **Collapsed-initials guard** — if either side contains `[A-Za-z]\.` (a letter+period), route to the deferred initial_vs_full bucket instead of treating as full-full
- **Pinyin / romanized-CJK first-name exclusion** — narrow token list; dropped ~26K same-first romanized-CJK candidates where Phase 2's 86% FP-rate caveat applied

**Two shipped lanes, unioned:**

| Lane | Predicate | Sampled precision |
|---|---|---:|
| `both_initial_diff` | Both middles = 1 letter, disagree | ~95% |
| `both_full_diff_diff_initial` | Both middles ≥2 chars, no substring overlap, different first letter, no letter-period, not a token reorder | ~92% |

**Deferred lanes (not shipped here):**
- `both_full_diff_same_initial` (~1.3M records, ~13% precision — blocked on parser-layer initials-expander)
- `initial_vs_full_diff` (~0.37M, ~60% precision — blocked on compound-surname-aware matcher for Iberian `de/da/di` tokens)
- `full_vs_initial_diff` (~13K, same FP risk as above)

Sampling (2 seeds × 25 rows × 2 lanes, v2 guards): **~210K total candidates, ~92–96% precision per lane**. Residual FP classes: bibliographic catalog records with embedded lifespans (`(1747-1825)`), initial-without-period runs (`Deepak MR Ayyala`), and near-threshold minority clusters (~29%). Rollback cell catches institution-overlap + close-middle cases after.

### Cell 1: Build candidates table

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_candidates_middle_name AS
WITH work_parsed AS (
  SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name,
    an_w.parsed_name.first  AS work_first,
    an_w.parsed_name.middle AS work_middle,
    an_w.parsed_name.last   AS work_last,
    -- Cluster size: works under this author sharing this (first, middle).
    -- Computed BEFORE the candidate WHERE filters so it reflects the profile's
    -- overall (first, middle) distribution, not just candidate-eligible rows.
    COUNT(*) OVER (
      PARTITION BY wa.author_id, an_w.parsed_name.first, an_w.parsed_name.middle
    ) AS cluster_works
  FROM openalex.works.work_authors wa
  JOIN openalex.authors.author_names an_w ON wa.raw_author_name = an_w.raw_author_name
  WHERE wa.author_id IS NOT NULL
    AND an_w.parsed_name.last   IS NOT NULL
    AND an_w.parsed_name.first  IS NOT NULL
    AND an_w.parsed_name.middle IS NOT NULL
    AND LENGTH(an_w.parsed_name.last)  > 2
    AND LENGTH(an_w.parsed_name.first) > 2
    AND LENGTH(TRIM(TRANSLATE(an_w.parsed_name.middle, '.', ''))) > 0
    -- CJK + Arabic script exclusion (wider than Phase 2's CJK-only range).
    AND NOT REGEXP_LIKE(wa.raw_author_name, '[가-힯぀-ヿ一-鿿؀-ۿ]')
),
-- Per-row work raw_orcid from source metadata (used for per-row ORCID exclusion).
work_orcids AS (
  SELECT wb.id AS work_id, pos AS author_sequence, authorship.raw_orcid
  FROM openalex.works.openalex_works_base wb
  LATERAL VIEW POSEXPLODE(wb.authorships) t AS pos, authorship
  WHERE authorship.raw_orcid IS NOT NULL AND authorship.raw_orcid != ''
),
-- Resolved institution IDs per (work_id, author_sequence). Surfaced for the
-- rollback cell, not used in the candidate filter.
work_inst_ids AS (
  SELECT work_id, author_sequence,
    COLLECT_SET(CONCAT('https://openalex.org/I', CAST(institution_id AS STRING))) AS work_institutions
  FROM openalex.works.work_author_affiliations
  WHERE institution_id IS NOT NULL
  GROUP BY work_id, author_sequence
),
base AS (
  SELECT wp.work_id, wp.author_sequence, wp.author_id, wp.raw_author_name,
    wp.work_first, wp.work_middle, wp.work_last, wp.cluster_works,
    ap.first AS author_first, ap.middle AS author_middle, ap.last AS author_last,
    ap.orcid, ap.full_name AS author_full_name, ap.works_count,
    COALESCE(wi.work_institutions, ARRAY()) AS work_institutions,
    COALESCE(ap.institution_ids, ARRAY()) AS author_institution_ids,
    TRANSLATE(LOWER(wp.work_middle), '. ', '') AS wm_norm,
    TRANSLATE(LOWER(ap.middle),      '. ', '') AS am_norm
  FROM work_parsed wp
  JOIN openalex.authors.authors_for_matching ap ON wp.author_id = ap.author_id
  LEFT JOIN work_inst_ids wi
    ON wp.work_id = wi.work_id AND wp.author_sequence = wi.author_sequence
  WHERE wp.work_first = ap.first
    AND wp.work_last  = ap.last
    AND ap.middle IS NOT NULL
    AND LENGTH(TRIM(TRANSLATE(ap.middle, '.', ''))) > 0
    AND NOT REGEXP_LIKE(ap.full_name, '[가-힯぀-ヿ一-鿿؀-ۿ]')
    AND NOT EXISTS (
      SELECT 1 FROM work_orcids wo
      WHERE wo.work_id = wp.work_id
        AND wo.author_sequence = wp.author_sequence
        AND wo.raw_orcid = ap.orcid
    )
    -- Profile parser-artifact guard. `[.]` char class for literal period;
    -- `\\[` / `\\]` (4 backslashes in Python source = 2 backslashes in
    -- the SQL literal = `\[` / `\]` regex after Spark strips one level).
    -- Java regex requires brackets escaped; `[[]` is invalid syntax.
    AND NOT REGEXP_LIKE(ap.middle, '\\[|\\]|[0-9]{3,}|AUTHOR|USGS|NULL|CA[.] UM')
    -- Minority-cluster guard.
    AND wp.cluster_works * 1.0 / NULLIF(ap.works_count, 0) <= 0.30
    -- Pinyin / romanized-CJK first-name exclusion.
    AND LOWER(wp.work_first) NOT IN ('akira','bao','bing','bo','byung','chao','chul','eun','feng','guo','hao','hiroshi','hong','hoon','hua','hui','hye','hyun','ji','jia','jie','jin','jong','joon','jun','jung','kang','ke','kenji','kun','kyung','lan','lei','li','lin','liu','min','ming','nan','ning','pei','peng','ping','qian','qiang','qin','qing','rui','satoshi','seong','seung','shan','sheng','shi','shu','shun','soo','suk','sung','taek','takashi','tao','tian','wei','wen','won','woong','xiang','xiao','xin','xue','yan','yi','ying','yong','yoshio','young','yu','zhao','zhe','zhen','zhi','zhong','zhou')
    AND LOWER(ap.first)      NOT IN ('akira','bao','bing','bo','byung','chao','chul','eun','feng','guo','hao','hiroshi','hong','hoon','hua','hui','hye','hyun','ji','jia','jie','jin','jong','joon','jun','jung','kang','ke','kenji','kun','kyung','lan','lei','li','lin','liu','min','ming','nan','ning','pei','peng','ping','qian','qiang','qin','qing','rui','satoshi','seong','seung','shan','sheng','shi','shu','shun','soo','suk','sung','taek','takashi','tao','tian','wei','wen','won','woong','xiang','xiao','xin','xue','yan','yi','ying','yong','yoshio','young','yu','zhao','zhe','zhen','zhi','zhong','zhou')
)
SELECT *
FROM (
  SELECT *,
    CASE
      WHEN LENGTH(wm_norm) = 1 AND LENGTH(am_norm) = 1 AND wm_norm <> am_norm
        THEN 'both_initial_diff'
      WHEN LENGTH(wm_norm) >= 2 AND LENGTH(am_norm) >= 2
        AND SUBSTRING(wm_norm, 1, 1) <> SUBSTRING(am_norm, 1, 1)
        AND INSTR(wm_norm, am_norm) = 0
        AND INSTR(am_norm, wm_norm) = 0
        AND SORT_ARRAY(SPLIT(LOWER(TRIM(work_middle)),   ' ')) <>
            SORT_ARRAY(SPLIT(LOWER(TRIM(author_middle)), ' '))
        -- Collapsed-initials guard: char class `[.]` = literal period.
        -- `[A-Za-z][.]` matches a letter immediately followed by a period
        -- (F.L., H.A.). Routes initial-runs to the deferred bucket instead
        -- of treating them as multi-char "full" middles.
        AND NOT REGEXP_LIKE(work_middle,   '[A-Za-z][.]')
        AND NOT REGEXP_LIKE(author_middle, '[A-Za-z][.]')
        THEN 'both_full_diff_diff_initial'
      ELSE NULL
    END AS lane
  FROM base
)
WHERE lane IS NOT NULL

### Cell 2: Validation statistics (per lane)

In [ ]:
SELECT
  lane,
  COUNT(*)                  AS total_candidates,
  COUNT(DISTINCT author_id) AS distinct_authors,
  COUNT(DISTINCT work_id)   AS distinct_works,
  PERCENTILE_APPROX(works_count, ARRAY(0.25, 0.5, 0.75, 0.95)) AS author_works_pctiles
FROM openalex.authors.overmerge_split_candidates_middle_name
GROUP BY lane
ORDER BY total_candidates DESC

### Cell 3: Spot-check sample (manual review)

In [ ]:
SELECT lane, work_id, author_id, raw_author_name, author_full_name,
  work_first, work_middle, author_middle, work_last,
  cluster_works, works_count AS author_works_count,
  ROUND(cluster_works * 1.0 / NULLIF(works_count, 0), 4) AS cluster_pct,
  orcid
FROM openalex.authors.overmerge_split_candidates_middle_name
ORDER BY RAND()
LIMIT 50

### Cell 4: Create audit log (for rollback)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_log_middle_name AS
SELECT work_id, author_sequence, author_id AS previous_author_id,
  raw_author_name, work_first, work_middle, author_middle, work_last,
  lane, current_timestamp() AS split_timestamp
FROM openalex.authors.overmerge_split_candidates_middle_name

### Cell 5: Execute the split

**WARNING**: This nulls out author_ids. Verify cells 2-3 before running.

**NOTE**: MatchAuthors has cutoffs (`work_id > 7B`, `created_date >= 2025-12-20`). Older split records will NOT be re-processed automatically — same blocker as Phase 1/2. Reuses whatever catch-up re-match solution lands for `PLAN-revisit-2026-04-20.md` Step 4.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT DISTINCT work_id, author_sequence
  FROM openalex.authors.overmerge_split_candidates_middle_name
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 6: Queue split works for reprocessing by UpdateWorkAuthorships

Push affected work_ids into `curated_work_ids_pending_sync` so the next `UpdateWorkAuthorships` run picks them up and propagates the nulled author_ids through to `work_authorships` → `openalex_works`.

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_middle_name

### Cell 7: Post-split verification

In [ ]:
SELECT
  COUNT(*) AS nulled_records,
  COUNT(DISTINCT c.work_id) AS works_affected
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_middle_name c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE wa.author_id IS NULL

### Cell 8: Outcome tracking (run after MatchAuthors re-processes)

Stratified by lane so we can see whether `both_initial_diff` and `both_full_diff_diff_initial` re-match at different rates.

In [ ]:
SELECT
  CASE
    WHEN wa.author_id IS NULL THEN 'STILL_UNMATCHED'
    WHEN wa.author_id = log.previous_author_id THEN 'RE_MATCHED_SAME'
    ELSE 'RE_MATCHED_NEW'
  END AS outcome,
  log.lane,
  COUNT(*) AS cnt
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_middle_name log
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
GROUP BY 1, 2
ORDER BY lane, outcome

### Cell 9: Rollback middle-name false positives — `both_full_diff_diff_initial` only

Restore where the work shares an institution with the profile AND the middle names are close. Scoped to the full-full lane only. Catches:

- **Spelling variants** — e.g. `Issoufou` / `Issouffou`, `Fioruci` / `Fiorucci` (LEVENSHTEIN ≤ 2 on normalized middles).
- **Parser-allocation drift** — one side's middle appears as substring of the other's middle or full_name.
- **Residual initials-vs-full leaks** — middle 'x.' slipped into `both_full_diff` because the collapsed-initials regex missed a variant.

**Why the lane guard:** for `both_initial_diff` (1-char middles that disagree), `LEVENSHTEIN = 1` trivially, and a 1-char INSTR against `full_name` matches whenever that letter appears anywhere in the profile. The previous lane-agnostic rollback therefore restored ~63% of the initial-diff lane on institution overlap alone — not the spelling-variant class it was meant to catch. The initial-diff lane's genuine FP class (same person, two different middle-initial renderings at one institution) is real but sits at the lane's ~4–6% FP rate per sampling, which the audit log + post-MatchAuthors outcome tracking handles adequately.

**Co-signal requirement:** institution overlap AND one of the three middle-closeness signals. Only updates rows still NULL (won't overwrite re-matches).

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT work_id, author_sequence, FIRST(author_id) AS previous_author_id
  FROM openalex.authors.overmerge_split_candidates_middle_name
  -- Lane guard: rollback only applies to the full-full lane, where
  -- LEVENSHTEIN <= 2 meaningfully catches spelling variants (Issoufou /
  -- Issouffou). For `both_initial_diff` (1-char middles that disagree),
  -- LEVENSHTEIN = 1 trivially, and a 1-char INSTR against full_name almost
  -- always hits, so this predicate would restore the whole lane whenever
  -- institutions overlap. That's not the intended rollback class.
  WHERE lane = 'both_full_diff_diff_initial'
    AND ARRAYS_OVERLAP(work_institutions, author_institution_ids)
    AND ( LEVENSHTEIN(wm_norm, am_norm) <= 2
       OR INSTR(LOWER(author_full_name), LOWER(TRIM(work_middle))) > 0
       OR INSTR(LOWER(TRIM(author_middle)), LOWER(TRIM(work_middle))) > 0
       OR INSTR(LOWER(TRIM(work_middle)), LOWER(TRIM(author_middle))) > 0 )
  GROUP BY work_id, author_sequence
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.author_id IS NULL THEN
  UPDATE SET
    target.author_id = source.previous_author_id,
    target.updated_at = current_timestamp()

### Cell 10: Queue restored works for reprocessing

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_middle_name
WHERE lane = 'both_full_diff_diff_initial'
  AND ARRAYS_OVERLAP(work_institutions, author_institution_ids)
  AND ( LEVENSHTEIN(wm_norm, am_norm) <= 2
     OR INSTR(LOWER(author_full_name), LOWER(TRIM(work_middle))) > 0
     OR INSTR(LOWER(TRIM(author_middle)), LOWER(TRIM(work_middle))) > 0
     OR INSTR(LOWER(TRIM(work_middle)), LOWER(TRIM(author_middle))) > 0 )

### Cell 11: Post-rollback verification

In [ ]:
SELECT
  COUNT(*) AS restored_records,
  COUNT(DISTINCT c.author_id) AS restored_authors
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_candidates_middle_name c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE c.lane = 'both_full_diff_diff_initial'
  AND ARRAYS_OVERLAP(c.work_institutions, c.author_institution_ids)
  AND ( LEVENSHTEIN(c.wm_norm, c.am_norm) <= 2
     OR INSTR(LOWER(c.author_full_name), LOWER(TRIM(c.work_middle))) > 0
     OR INSTR(LOWER(TRIM(c.author_middle)), LOWER(TRIM(c.work_middle))) > 0
     OR INSTR(LOWER(TRIM(c.work_middle)), LOWER(TRIM(c.author_middle))) > 0 )
  AND wa.author_id = c.author_id

### Cell 12: Fix-up — re-null `both_initial_diff` rows restored by the lane-agnostic rollback

**Only needed for notebooks that executed Cell 9 before the lane guard was added (pre-2026-04-22).** On a fresh run of the patched Cell 9, this cell is a safe no-op because the lane-agnostic restoration never happens.

Matches on `target.author_id = previous_author_id` so rows that MatchAuthors has already reassigned to a different author (`RE_MATCHED_NEW`) are preserved — we only undo restorations that put the row back on its pre-split profile.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT log.work_id, log.author_sequence, log.previous_author_id
  FROM openalex.authors.overmerge_split_log_middle_name log
  WHERE log.lane = 'both_initial_diff'
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.author_id = source.previous_author_id THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 13: Queue re-nulled works for reprocessing

UpdateWorkAuthorships already propagated the (buggy) restored author_ids to `work_authorships` / `openalex_works`. Queue the re-nulled work_ids so the next UpdateWorkAuthorships run propagates the NULL back through.

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT log.work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_log_middle_name log
JOIN openalex.works.work_authors wa
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
WHERE log.lane = 'both_initial_diff'
  AND wa.author_id IS NULL

### Cell 14: Post-renull verification (both_initial_diff lane only)

Expect: most rows in `RE_NULLED_OK`. A small `RE_MATCHED_NEW` bucket is fine (MatchAuthors found a better match for that work). `STILL_RESTORED_TO_PREVIOUS` should be zero; any residual here means the re-null MERGE didn't reach that row.

In [ ]:
SELECT
  CASE
    WHEN wa.author_id IS NULL THEN 'RE_NULLED_OK'
    WHEN wa.author_id = log.previous_author_id THEN 'STILL_RESTORED_TO_PREVIOUS'
    ELSE 'RE_MATCHED_NEW'
  END AS outcome,
  COUNT(*) AS cnt
FROM openalex.authors.overmerge_split_log_middle_name log
JOIN openalex.works.work_authors wa
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
WHERE log.lane = 'both_initial_diff'
GROUP BY 1
ORDER BY 1

### Cell 15: Recovery — append `both_full_diff_diff_initial` lane to audit log

**Only needed if Cell 1 was run before the regex fix (pre-2026-04-22 evening).** After re-running Cell 1 with the patched regex, the candidates table now has ~210K rows across both lanes. The audit log from the earlier run has only the `both_initial_diff` 66K rows. MERGE appends the missing full-full rows without touching existing entries (so `previous_author_id` for initial_diff stays captured).

In [ ]:
MERGE INTO openalex.authors.overmerge_split_log_middle_name AS target
USING (
  SELECT work_id, author_sequence, author_id AS previous_author_id,
    raw_author_name, work_first, work_middle, author_middle, work_last,
    lane, current_timestamp() AS split_timestamp
  FROM openalex.authors.overmerge_split_candidates_middle_name
  WHERE lane = 'both_full_diff_diff_initial'
) AS source
ON target.work_id = source.work_id AND target.author_sequence = source.author_sequence
WHEN NOT MATCHED THEN
  INSERT (work_id, author_sequence, previous_author_id,
    raw_author_name, work_first, work_middle, author_middle, work_last,
    lane, split_timestamp)
  VALUES (source.work_id, source.author_sequence, source.previous_author_id,
    source.raw_author_name, source.work_first, source.work_middle,
    source.author_middle, source.work_last, source.lane, source.split_timestamp)

### Cell 16: Recovery — null `both_full_diff_diff_initial` rows in work_authors

Null only rows where `target.author_id = source.author_id` (the pre-split author_id captured in the candidates table). This makes the merge safe to re-run and avoids clobbering any row that `MatchAuthors` may have already moved off the pre-split profile in the meantime.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT DISTINCT work_id, author_sequence, author_id
  FROM openalex.authors.overmerge_split_candidates_middle_name
  WHERE lane = 'both_full_diff_diff_initial'
) AS source
ON target.work_id = source.work_id AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.author_id = source.author_id THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 17: Recovery — queue `both_full_diff_diff_initial` works for reprocessing

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_middle_name
WHERE lane = 'both_full_diff_diff_initial'

### Cell 18: Recovery — verification

Per-lane state of the log after recovery. Expected:

- `both_initial_diff`: ~66K (preserved from original run)
- `both_full_diff_diff_initial`: ~144K (just appended)
- `total nulled` in work_authors across both lanes: ~210K

In [ ]:
SELECT lane,
  COUNT(*) AS log_rows,
  SUM(CASE WHEN wa.author_id IS NULL THEN 1 ELSE 0 END) AS currently_nulled
FROM openalex.authors.overmerge_split_log_middle_name log
JOIN openalex.works.work_authors wa
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
GROUP BY lane
ORDER BY lane